# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided workflow for loading and exploring the FAIR² dataset using the `mlcroissant` library. In each section, we'll use Croissant schema `@id` values to reference record sets, fields, and columns according to schema best practices.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Display overall dataset information
ds_metadata = dataset.metadata  # mlcroissant.Metadata object
print(f"Title: {ds_metadata.name}")
print(f"Description: {ds_metadata.description}")
print(f"Version: {ds_metadata.version}  |  License: {ds_metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their Croissant `@id`s for reference throughout analysis.

In [ ]:
# Display available record sets
print("Available Record Sets (by @id):")
record_sets = list(ds_metadata.record_sets)
for rs in record_sets:
    print(f" - {rs['@id']}: {rs.get('name')}")

# For each record set, display the fields/columns
print("\nFields (by @id) within each Record Set:")
record_set_fields = {}
for rs in record_sets:
    print(f"\nRecordSet @id: {rs['@id']}  |  Name: {rs.get('name')}")
    fields = rs.get('field', [])
    # Ensure fields is always a list
    if isinstance(fields, dict):
        fields = [fields]
    record_set_fields[rs['@id']] = fields
    for field in fields:
        print(f"  - @id: {field['@id']}", end='')
        if 'name' in field:
            print(f" | Name: {field['name']}")
        else:
            print()

# Example: Show first 2 records for each record set (usually one primary)
for rs in record_sets:
    print(f"\nFirst 2 records from RecordSet @id: {rs['@id']}")
    try:
        for i, rec in enumerate(dataset.records(record_set=rs['@id'])):
            print(rec)
            if i >= 1:
                break
    except Exception as e:
        print(f"Could not read records: {e}")

## 3. Data Extraction
Load tabular data from the main record set(s) into pandas DataFrames for analysis. All entity references use their Croissant `@id`.

In [ ]:
# List record set @ids to load
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame for '{record_set_id}' with shape {dataframes[record_set_id].shape}")
            print("Columns:", dataframes[record_set_id].columns.tolist())
            display(dataframes[record_set_id].head(2))
        else:
            print(f"No records found for '{record_set_id}'")
    except Exception as e:
        print(f"Failed loading '{record_set_id}': {e}")

# Pick a specific record set for analysis (choose main tabular data)
# (Change this to the @id you wish to analyze; often the first in the list)
main_record_set_id = record_set_ids[0] if record_set_ids else None

if main_record_set_id and main_record_set_id in dataframes:
    print(f"\nColumn names in '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records by certain numeric field criteria, normalizing, and grouping. All references are to fields by `@id`.

In [ ]:
# Replace with a numeric field @id from the field overview, e.g. a quantification or abundance column
# Use actual @id as per the previous exploration
numeric_field_id = None

# Try to auto-detect a numeric field for demonstration
if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

if not numeric_field_id:
    print("No numeric field found for demonstration.")
else:
    print(f"Using numeric field: {numeric_field_id}")

    # Define a threshold for filtering
    threshold = df[numeric_field_id].mean()  # Use mean as a default

    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} (>{threshold:.2f}): {len(filtered_df)} rows")
    display(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Try grouping if a categorical field is available
    # Find a likely group field (first object dtype column after index)
    group_field_id = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
            group_field_id = col
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean of {numeric_field_id} grouped by '{group_field_id}':")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes and numeric_field_id:
    # Histogram of the numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If group_field_id exists, boxplot
    if group_field_id:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[main_record_set_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, process, and visualize the FAIR² mass spectrometry dataset using the `mlcroissant` library, always referencing fields and record sets by their Croissant schema `@id`s. 

You can extend this analysis to more fields and record sets for deeper biological or data science questions!